# Full MuxVizPy analysis with real data

Four complete protein-interaction layers from
[Zambelli et al. (2025)](https://doi.org/10.3390/e27121248) provide real data for the
main MuxVizPy features. All 56,534 source interactions are kept.

In [ ]:
from pathlib import Path

import graph_tool as gt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import display

from MuxVizPy import (
    decomposition,
    global_descriptors,
    information,
    mesoscale,
    percolation,
    topology,
    versatility,
    visualization,
)
from MuxVizPy.utils import parsing

np.random.seed(42)
gt.seed_rng(42)
plt.rcParams["figure.dpi"] = 110

In [ ]:
def labelled_heatmap(matrix, xlabels, ylabels, title, ax, fmt=".2f", cmap="viridis"):
    ax.imshow(matrix, cmap=cmap)
    ax.set_xticks(range(len(xlabels)), xlabels, rotation=35, ha="right")
    ax.set_yticks(range(len(ylabels)), ylabels)
    ax.set_title(title)
    for row in range(matrix.shape[0]):
        for col in range(matrix.shape[1]):
            value = format(matrix[row, col], fmt)
            ax.text(col, row, value, ha="center", va="center")

## Load the layers

Each undirected edge list is checked for missing or repeated rows.

In [ ]:
DATA_DIR = Path("notebooks/data/oncovirus")
if not DATA_DIR.exists():
    DATA_DIR = Path("data/oncovirus")

VIRUSES = {
    "Gallid herpesvirus 2": "gallid-herpesvirus-2.csv",
    "Yaba monkey tumour virus": "yaba-monkey-tumour-virus.csv",
    "Equine herpesvirus 2": "equine-herpesvirus-2.csv",
    "Human parechovirus 2": "human-parechovirus-2.csv",
}
SHORT_NAMES = ["Gallid", "Yaba", "Equine", "Parechovirus"]
layers = {
    virus: pl.read_csv(DATA_DIR / filename).select("source", "target")
    for virus, filename in VIRUSES.items()
}
for edges in layers.values():
    assert edges.null_count().sum_horizontal().item() == 0
    assert edges.unique().height == edges.height

In [ ]:
layer_summary = pl.DataFrame(
    {
        "layer": SHORT_NAMES,
        "active proteins": [
            len(set(edges["source"]) | set(edges["target"]))
            for edges in layers.values()
        ],
        "interactions": [edges.height for edges in layers.values()],
        "self-interactions": [
            edges.filter(pl.col("source") == pl.col("target")).height
            for edges in layers.values()
        ],
    }
)
layer_summary

## Network forms

Each protein receives one shared index. The network forms are a sparse `(N, L, N, L)`
tensor, graph list, layer matrices, edge-coloured supra matrix, coupled supra matrix,
and aggregate matrix.

In [ ]:
proteins = sorted(
    set().union(
        *(set(edges["source"]) | set(edges["target"]) for edges in layers.values())
    )
)
node_id = {protein: index for index, protein in enumerate(proteins)}
N, L = len(proteins), len(layers)

directed_rows = []
for layer_index, edges in enumerate(layers.values()):
    source = np.array([node_id[name] for name in edges["source"]])
    target = np.array([node_id[name] for name in edges["target"]])
    forward = pl.DataFrame(
        {
            "node.from": source,
            "layer.from": np.full(len(source), layer_index),
            "node.to": target,
            "layer.to": np.full(len(source), layer_index),
            "weight": np.ones(len(source)),
        }
    )
    reverse = forward.filter(pl.col("node.from") != pl.col("node.to")).select(
        pl.col("node.to").alias("node.from"),
        pl.col("layer.to").alias("layer.from"),
        pl.col("node.from").alias("node.to"),
        pl.col("layer.from").alias("layer.to"),
        "weight",
    )
    directed_rows.extend([forward, reverse])

extended_edges = pl.concat(directed_rows)
tensor = parsing.build_tensor_from_dataframe(extended_edges)
g_list = parsing.build_list_of_graphs_from_tensor(tensor, directed=False)
node_tensor = parsing.get_node_tensor_from_network_list(g_list)
edge_coloured = parsing.build_supra_adjacency_matrix_from_tensor(tensor)
aggregate = parsing.build_aggregate_network_from_tensor(tensor)
coupling = parsing.build_interlayer_coupling_matrix(L, 1.0, "categorical")
coupled = parsing.build_supra_adjacency_matrix_from_edge_colored_matrices(
    node_tensor, coupling, N
)

In [ ]:
expected_edges = layer_summary["interactions"].to_list()
assert [graph.num_edges() for graph in g_list] == expected_edges
assert tensor.shape == (N, L, N, L)
assert edge_coloured.shape == coupled.shape == (N * L, N * L)

network_forms = pl.DataFrame(
    {
        "form": ["tensor", "edge-coloured supra", "coupled supra", "aggregate"],
        "shape": [
            str(tuple(tensor.shape)), str(edge_coloured.shape),
            str(coupled.shape), str(aggregate.shape),
        ],
        "stored values": [
            tensor._nnz(), edge_coloured.nnz, coupled.nnz, aggregate.nnz,
        ],
    }
)
print(f"{N:,} proteins, {L} layers, {sum(expected_edges):,} interactions")
network_forms

The edge-coloured matrix contains observed links. The coupled matrix also connects each
protein to its replicas in other layers. Layer comparisons use the first; paths and
centralities that change layer use the second.

## Layer structure

In [ ]:
active_sets = []
density = []
component_counts = []
for graph in g_list:
    degree_in_layer = graph.get_total_degrees(graph.get_vertices())
    active = set(np.flatnonzero(degree_in_layer > 0))
    active_sets.append(active)
    possible = len(active) * (len(active) - 1) / 2
    density.append(graph.num_edges() / possible)
    component_labels, _ = gt.topology.label_components(graph)
    labels = component_labels.get_array()[degree_in_layer > 0]
    component_counts.append(int(np.unique(labels).size))

layer_structure = layer_summary.with_columns(
    pl.Series("density", density),
    pl.Series("connected components", component_counts),
)
layer_structure

In [ ]:
layer_membership = np.array(
    [sum(index in active for active in active_sets) for index in range(N)]
)
membership_counts = pl.DataFrame(
    {
        "number of layers": np.arange(1, L + 1),
        "proteins": [
            np.count_nonzero(layer_membership == count)
            for count in range(1, L + 1)
        ],
    }
)
shared_active = np.array(
    [[len(first & second) for second in active_sets] for first in active_sets]
)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].bar(membership_counts["number of layers"], membership_counts["proteins"])
axes[0].set(xlabel="number of active layers", ylabel="proteins", title="Layer presence")
labelled_heatmap(
    shared_active, SHORT_NAMES, SHORT_NAMES, "Shared active proteins", axes[1], fmt="d"
)
fig.tight_layout()
plt.show()
membership_counts

## Multilayer components

`get_multi_LCC`, `get_multi_LIC`, and `get_multi_LVC` return the largest connected,
intersected, and viable components.

In [ ]:
lcc = np.asarray(topology.get_multi_LCC(g_list), dtype=int)
lic = np.asarray(topology.get_multi_LIC(g_list), dtype=int)
lvc = np.asarray(topology.get_multi_LVC(g_list, printt=False), dtype=int)

component_summary = pl.DataFrame(
    {
        "component": ["largest connected", "largest intersected", "largest viable"],
        "proteins": [len(lcc), len(lic), len(lvc)],
    }
)
print("Viable component:", ", ".join(proteins[index] for index in lvc))
component_summary

## Node importance

The node scores are aggregate degree, eigenvector, Katz, PageRank, K-core, and an
edge-coloured random walk. For Katz, `rho` is the coupled matrix's spectral radius, and
`alpha = 0.5 / rho` keeps its scores distinct from eigenvector centrality.

In [ ]:
degree = versatility.get_multi_degree(edge_coloured, nodes=N, layers=L)
eigenvector = versatility.compute_eigenvector_centrality(
    coupled, nodes=N, layers=L
)
katz_rho, _ = versatility.get_largest_eigenvalue(coupled)
katz_alpha = 0.5 / katz_rho
katz = versatility.compute_katz_centrality(
    coupled, nodes=N, layers=L, alpha=katz_alpha, solver="bicgstab"
)
pagerank = versatility.compute_multipagerank_centrality(
    coupled, nodes=N, layers=L
)
kcore = versatility.get_multi_Kcore_centrality(
    edge_coloured, nodes=N, layers=L
)
edge_rw_frame = versatility.get_multi_RW_centrality_edge_colored(node_tensor)
edge_rw = np.zeros(N)
edge_rw[edge_rw_frame["phy nodes"].to_numpy(dtype=int)] = edge_rw_frame[
    "vers"
].to_numpy()

scores = pd.DataFrame(
    {
        "protein": proteins, "degree": degree, "eigenvector": eigenvector,
        "katz": katz, "pagerank": pagerank, "k-core": kcore,
        "edge-coloured random walk": edge_rw,
    }
).set_index("protein")

In [ ]:
top_ranked = pd.DataFrame(
    {measure: scores[measure].nlargest(10).index for measure in scores.columns}
)
rank_correlation = scores.corr(method="spearman").to_numpy()
fig, ax = plt.subplots(figsize=(7, 6))
labelled_heatmap(
    rank_correlation, scores.columns, scores.columns,
    "Spearman correlation between node rankings", ax, cmap="coolwarm"
)
fig.tight_layout()
plt.show()
top_ranked

## Layer comparison

The layer comparisons cover shared edges, shared active nodes, degree correlation,
shortest paths, entropy, and Jensen-Shannon divergence.

In [ ]:
edge_overlap = global_descriptors.compute_average_global_overlap_matrix(
    edge_coloured, nodes=N, layers=L
)
node_overlap = global_descriptors.compute_average_global_node_overlap_matrix(
    edge_coloured, nodes=N, layers=L
)
assortativity = mesoscale.inter_layer_assortativity(g_list, layers=L)
path_similarity = topology.get_SP_similarity_matrix(
    edge_coloured, nodes=N, layers=L
)

entropies = []
active_masks = []
for graph, adjacency in zip(g_list, node_tensor, strict=True):
    active = graph.get_total_degrees(graph.get_vertices()) > 0
    active_masks.append(active)
    reduced = adjacency[active][:, active]
    density_matrix = parsing.build_density_bgs_from_adjacency_matrix(reduced)
    entropies.append(information.compute_vn_entropy(density_matrix))

js_divergence = np.zeros((L, L))
for first in range(L):
    for second in range(first + 1, L):
        active_union = active_masks[first] | active_masks[second]
        first_adj = node_tensor[first][active_union][:, active_union]
        second_adj = node_tensor[second][active_union][:, active_union]
        value = information.compute_js_divergence(
            first_adj, second_adj, entropies[first], entropies[second]
        )
        js_divergence[first, second] = js_divergence[second, first] = value

In [ ]:
comparison_matrices = [
    (edge_overlap, "Edge overlap", "viridis"),
    (node_overlap, "Active-node overlap", "viridis"),
    (assortativity["Pearson"], "Pearson degree correlation", "coolwarm"),
    (assortativity["Spearman"], "Spearman degree correlation", "coolwarm"),
    (path_similarity, "Shortest-path similarity", "viridis"),
    (js_divergence, "Jensen-Shannon divergence", "magma"),
]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, (matrix, title, cmap) in zip(axes.flat, comparison_matrices, strict=True):
    labelled_heatmap(matrix, SHORT_NAMES, SHORT_NAMES, title, ax, cmap=cmap)
fig.tight_layout()
plt.show()
entropy_table = pl.DataFrame(
    {"layer": SHORT_NAMES, "Von Neumann entropy": entropies}
)
entropy_table

Entropy uses each layer's active nodes. Pairwise divergence uses the same active-node
union for both layers.

## Clustering and paths

Global clustering, physical-node distances, average path length, and closeness are
calculated. Paths use the coupled matrix so they can change layer.

In [ ]:
clustering_observed = (
    global_descriptors.compute_average_global_clustering_coefficient(
        edge_coloured, nodes=N, layers=L
    )
)
clustering_coupled = (
    global_descriptors.compute_average_global_clustering_coefficient(
        coupled, nodes=N, layers=L
    )
)
path_statistics = topology.get_multi_path_statistics(
    coupled, nodes=N, layers=L
)
closeness = np.asarray(path_statistics["closeness"])
path_summary = pl.DataFrame(
    {
        "result": [
            "global clustering, observed", "global clustering, coupled",
            "average multilayer path length",
        ],
        "value": [
            clustering_observed, clustering_coupled,
            path_statistics["avg_path_length"],
        ],
    }
)
top_closeness = pd.DataFrame(
    {"protein": proteins, "closeness": closeness}
).nlargest(10, "closeness")
display(path_summary)
top_closeness

Source self-interactions make `compute_local_clustering_coefficient` reject the data, so
local clustering is omitted.

## Node-removal robustness

`get_percolation` compares removal by random-walk score with a fixed random order. Its
critical fraction is the peak of the second-largest component curve.

In [ ]:
ranked_nodes = np.argsort(edge_rw)[::-1]
random_nodes = np.random.default_rng(42).permutation(N)
targeted_attack = percolation.get_percolation(
    g_list, nodes=N, layers=L, order=ranked_nodes
)
random_attack = percolation.get_percolation(
    g_list, nodes=N, layers=L, order=random_nodes
)

fraction_removed = np.arange(N) / N
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, result, title in [
    (axes[0], targeted_attack, "Versatility order"),
    (axes[1], random_attack, "Fixed random order"),
]:
    ax.plot(fraction_removed, result["1ComponentSize"] / N, label="largest")
    ax.plot(fraction_removed, result["2ComponentSize"] / N, label="second-largest")
    ax.axvline(result["CritPoint"], color="black", linestyle=":")
    ax.set(xlabel="fraction removed", title=title)
    ax.grid(alpha=0.25)
axes[0].set_ylabel("fraction of all proteins")
axes[1].legend()
fig.tight_layout()
plt.show()

percolation_summary = pl.DataFrame(
    {
        "order": ["edge-coloured random walk", "fixed random"],
        "critical fraction": [
            targeted_attack["CritPoint"], random_attack["CritPoint"]
        ],
    }
)
percolation_summary

## Layered communities

`get_mod` fits a layered block model and reports its groups and modularity. The fit uses
random choices, so another run can differ.

In [ ]:
g_multi = gt.Graph(directed=False)
g_multi.add_vertex(N)
layer_property = g_multi.new_edge_property("int")
layered_edges = []
for layer_index, edges in enumerate(layers.values()):
    source = [node_id[name] for name in edges["source"]]
    target = [node_id[name] for name in edges["target"]]
    edge_layer = np.full(len(source), layer_index)
    layered_edges.append(np.column_stack((source, target, edge_layer)))
g_multi.add_edge_list(np.vstack(layered_edges), eprops=[layer_property])
g_multi.ep["weight"] = layer_property

module_counts, modularities = mesoscale.get_mod(g_multi, n_iter=1)
community_summary = pl.DataFrame(
    {"non-empty groups": module_counts, "modularity": modularities}
)
community_summary

## Tensor decomposition

A rank-3 sparse CP decomposition extracts three node and layer patterns.

In [ ]:
factors, component_weights, cp_history = decomposition.sparse_cp_decomposition(
    tensor, rank=3, init="random", max_iter=20,
    random_state=42, backend="numpy"
)
source_nodes, source_layers, target_nodes, target_layers = factors
node_loadings = np.sqrt(np.abs(source_nodes * target_nodes))
layer_loadings = np.sqrt(np.abs(source_layers * target_layers))
layer_pattern_table = pd.DataFrame(
    layer_loadings, index=SHORT_NAMES,
    columns=[f"pattern {index}" for index in range(1, 4)]
)
top_pattern_proteins = pd.DataFrame(
    {
        f"pattern {index + 1}": [
            proteins[node]
            for node in np.argsort(node_loadings[:, index])[-10:][::-1]
        ]
        for index in range(3)
    }
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
labelled_heatmap(
    layer_loadings, ["P1", "P2", "P3"], SHORT_NAMES,
    "Layer weights", axes[0]
)
axes[1].plot(cp_history["reconstruction_error"], marker="o", markersize=3)
axes[1].set(
    xlabel="iteration", ylabel="relative reconstruction error",
    title="CP convergence"
)
fig.tight_layout()
plt.show()
display(layer_pattern_table)
top_pattern_proteins

## Multiplex plot

`plotMultiplex` draws the top 30 proteins on four labelled planes. Only the drawing is
filtered; every calculation uses the full data.

In [ ]:
selected = np.argsort(edge_rw)[-30:][::-1]
selected_mask = np.zeros(N, dtype=bool)
selected_mask[selected] = True
plot_layers = [
    gt.Graph(gt.GraphView(graph, vfilt=selected_mask), prune=True)
    for graph in g_list
]
plot_aggregate = parsing.get_aggregate_network(plot_layers, obj_type="glist")
visualization.plotMultiplex(
    plot_layers, plot_aggregate, layer_labels=SHORT_NAMES,
    max_edges_per_layer=700, edge_alpha=0.18, layer_spacing=0.7
)
pl.DataFrame(
    {
        "plot index": np.arange(len(selected)),
        "protein": [proteins[index] for index in selected],
    }
)

## Saved results

In [ ]:
results = pl.DataFrame(
    {
        "result": [
            "physical proteins", "layers", "source interactions",
            "largest connected component", "largest intersected component",
            "largest viable component", "observed global clustering",
            "coupled average path length", "targeted critical fraction",
            "random critical fraction", "layered community groups",
        ],
        "value": [
            f"{N:,}", str(L), f"{sum(expected_edges):,}", str(len(lcc)),
            str(len(lic)), str(len(lvc)), f"{clustering_observed:.4f}",
            f"{path_statistics['avg_path_length']:.4f}",
            f"{targeted_attack['CritPoint']:.4f}",
            f"{random_attack['CritPoint']:.4f}", str(module_counts[0]),
        ],
    }
)
results